# Cross-Community Shift Cluster-Robust: Non-independence Robustness (R2-3.1)

Re-tests the pre-shift vs post-shift change in the share of cross-community interactions (Table 3, chi2 = 40.26) while accounting for dependence among interactions:
1. event-level logistic regression `is_cross ~ post` with no clustering, and with cluster-robust SEs by source account, target account, both (two-way), and source community;
2. cluster bootstrap over source accounts (B = 10,000, seed 42);
3. the same comparison restricted to interactions originating in the largest connected component, where every cross-community interaction occurs;
4. conditional logistic regression with source-account fixed effects for accounts active in both periods.

Windows follow Table 3: pre-shift 1 to 26 August, post-shift 28 August to 29 September, 27 August excluded.

Input (`derived-data/hashed/`): `temporal_edges_with_community_hashed.csv`, `network_nodes_with_community_hashed.csv`, `network_edges_hashed.csv`. Reproduces Table S14.

In [1]:
# !pip install statsmodels  # uncomment if statsmodels is missing
import numpy as np, pandas as pd, networkx as nx, statsmodels.api as sm, warnings
from statsmodels.discrete.conditional_models import ConditionalLogit
from scipy.stats import chi2_contingency
warnings.filterwarnings('ignore')
ev = pd.read_csv('temporal_edges_with_community_hashed.csv')               # derived-data/hashed/
nodes = pd.read_csv('network_nodes_with_community_hashed.csv', sep=';')     # derived-data/hashed/
edges = pd.read_csv('network_edges_hashed.csv')                             # derived-data/hashed/
ev['date_day'] = pd.to_datetime(ev['date_day'])
w = ev[ev['date_day'] != '2025-08-27'].copy()                                # 27 August excluded, as in Table 3
w['post'] = (w['date_day'] >= '2025-08-28').astype(int)
w['y'] = w['is_cross_community'].astype(int)
t = pd.crosstab(w['post'], w['y']).rename(index={0: 'pre-shift', 1: 'post-shift'}, columns={0: 'within', 1: 'cross'})
t['total'] = t.sum(1); t['% cross'] = (100 * t['cross'] / t['total']).round(2); print(t)
chi2, p, dof, _ = chi2_contingency(t[['within', 'cross']].values, correction=True)
print(f'Table 3: chi2 (Yates) = {chi2:.2f}, df = {dof}, p = {p:.2e} | events = {len(w):,}, source accounts = {w.source.nunique():,}, '
      f'target accounts = {w.target.nunique():,}, source communities = {w.source_community.nunique():,}')

y           within  cross  total  % cross
post                                     
pre-shift     9225   1445  10670    13.54
post-shift    8618   1726  10344    16.69
Table 3: chi2 (Yates) = 40.26, df = 1, p = 2.23e-10 | events = 21,014, source accounts = 7,034, target accounts = 4,671, source communities = 1,651


In [2]:
# (1) event-level logistic regression, alternative treatments of dependence
X = sm.add_constant(w[['post']]); rows = []
g = {k: pd.factorize(w[k])[0] for k in ['source', 'target', 'source_community']}
specs = [('All events, no clustering', {}),
         ('All events, clustered by source account', {'cov_type': 'cluster', 'cov_kwds': {'groups': g['source']}}),
         ('All events, clustered by target account', {'cov_type': 'cluster', 'cov_kwds': {'groups': g['target']}}),
         ('All events, two-way clustered (source and target)', {'cov_type': 'cluster', 'cov_kwds': {'groups': np.column_stack([g['source'], g['target']])}}),
         ('All events, clustered by source community', {'cov_type': 'cluster', 'cov_kwds': {'groups': g['source_community']}})]
for name, kw in specs:
    m = sm.Logit(w['y'], X).fit(disp=0, **kw); lo, hi = np.exp(m.conf_int().loc['post'])
    rows.append([name, len(w), np.exp(m.params['post']), lo, hi, m.pvalues['post']])
    print(f'{name:<52} OR {np.exp(m.params["post"]):.3f}  95% CI [{lo:.3f}, {hi:.3f}]  SE {m.bse["post"]:.4f}  p {m.pvalues["post"]:.2e}')

All events, no clustering                            OR 1.279  95% CI [1.185, 1.379]  SE 0.0387  p 2.09e-10
All events, clustered by source account              OR 1.279  95% CI [1.018, 1.606]  SE 0.1163  p 3.45e-02
All events, clustered by target account              OR 1.279  95% CI [1.124, 1.454]  SE 0.0657  p 1.83e-04
All events, two-way clustered (source and target)    OR 1.279  95% CI [1.011, 1.617]  SE 0.1198  p 4.02e-02
All events, clustered by source community            OR 1.279  95% CI [0.975, 1.677]  SE 0.1384  p 7.57e-02


In [3]:
# (2) cluster bootstrap over source accounts (percentile CI for the difference and the odds ratio)
B = 10000; rng = np.random.default_rng(42)
agg = w.groupby(['source', 'post'])['y'].agg(['sum', 'count']).unstack(fill_value=0)
c0_, n0_, c1_, n1_ = (agg[('sum', 0)].values, agg[('count', 0)].values, agg[('sum', 1)].values, agg[('count', 1)].values)
K = len(agg); diffs = np.empty(B); ors = np.empty(B)
for b in range(B):
    k = np.bincount(rng.integers(0, K, K), minlength=K)
    q1, q0 = (k * c1_).sum() / (k * n1_).sum(), (k * c0_).sum() / (k * n0_).sum()
    diffs[b] = q1 - q0; ors[b] = (q1 / (1 - q1)) / (q0 / (1 - q0))
p1, p0 = c1_.sum() / n1_.sum(), c0_.sum() / n0_.sum(); or_obs = (p1 / (1 - p1)) / (p0 / (1 - p0))
lo, hi = np.percentile(ors, [2.5, 97.5])
rows.append(['All events, cluster bootstrap over source accounts (B = 10,000)', len(w), or_obs, lo, hi, np.nan])
print(f'source accounts = {K:,} | difference = {100*(p1-p0):.2f} pp, 95% CI [{100*np.percentile(diffs, 2.5):.2f}, {100*np.percentile(diffs, 97.5):.2f}] '
      f'| OR = {or_obs:.3f}, 95% CI [{lo:.3f}, {hi:.3f}] | bootstrap share with difference <= 0: {(diffs <= 0).mean():.4f}')

source accounts = 7,034 | difference = 3.14 pp, 95% CI [0.31, 6.04] | OR = 1.279, 95% CI [1.024, 1.614] | bootstrap share with difference <= 0: 0.0142


In [4]:
# (3) largest connected component: where every cross-community interaction occurs
G = nx.Graph(); G.add_nodes_from(nodes['username']); G.add_edges_from(zip(edges['source'], edges['target']))
lcc = max(nx.connected_components(G), key=len)
w['lcc'] = w['source'].isin(lcc).astype(int)
print(f'LCC: {len(lcc):,} nodes, {nodes[nodes.username.isin(lcc)].community.nunique()} communities | cross-community events outside LCC: {w.loc[w.lcc == 0, "y"].sum()}')
s = w.groupby('post')['lcc'].mean(); print(f'share of events from LCC: pre {100*s[0]:.1f}%, post {100*s[1]:.1f}%')
wl = w[w['lcc'] == 1]; s = wl.groupby('post')['y'].agg(['sum', 'count'])
print(f'cross-community share within LCC: pre {s.loc[0, "sum"]:,}/{s.loc[0, "count"]:,} = {100*s.loc[0, "sum"]/s.loc[0, "count"]:.2f}%, '
      f'post {s.loc[1, "sum"]:,}/{s.loc[1, "count"]:,} = {100*s.loc[1, "sum"]/s.loc[1, "count"]:.2f}%')
m = sm.Logit(wl['y'], sm.add_constant(wl[['post']])).fit(disp=0, cov_type='cluster', cov_kwds={'groups': pd.factorize(wl['source'])[0]})
lo, hi = np.exp(m.conf_int().loc['post'])
rows.append(['Largest connected component only, clustered by source account', len(wl), np.exp(m.params['post']), lo, hi, m.pvalues['post']])
print(f'LCC only, clustered by source account: OR {np.exp(m.params["post"]):.3f}  95% CI [{lo:.3f}, {hi:.3f}]  p {m.pvalues["post"]:.3f}')

LCC: 7,137 nodes, 52 communities | cross-community events outside LCC: 0
share of events from LCC: pre 72.1%, post 84.7%
cross-community share within LCC: pre 1,445/7,692 = 18.79%, post 1,726/8,763 = 19.70%
LCC only, clustered by source account: OR 1.060  95% CI [0.857, 1.312]  p 0.589


In [5]:
# (4) within-account change: conditional logit with source-account fixed effects
both = w.groupby('source')['post'].nunique(); both = both[both == 2].index
wb = w[w['source'].isin(both)]; wi = wb[wb.groupby('source')['y'].transform('nunique') == 2]
cl = ConditionalLogit(wi['y'].values, wi[['post']].values, groups=pd.factorize(wi['source'])[0]).fit(disp=0)
b, se = cl.params[0], cl.bse[0]
rows.append(['Accounts active in both periods, account fixed effects', len(wi), np.exp(b), np.exp(b - 1.96 * se), np.exp(b + 1.96 * se), cl.pvalues[0]])
print(f'accounts active in both periods: {len(both)} ({len(wb):,} events) | informative accounts: {wi.source.nunique()} ({len(wi):,} events)')
print(f'conditional logit: OR {np.exp(b):.3f}  95% CI [{np.exp(b - 1.96*se):.3f}, {np.exp(b + 1.96*se):.3f}]  p {cl.pvalues[0]:.3f}')

accounts active in both periods: 283 (4,131 events) | informative accounts: 193 (3,659 events)
conditional logit: OR 1.134  95% CI [0.942, 1.364]  p 0.183


In [6]:
# Table S14
s14 = pd.DataFrame(rows, columns=['Specification', 'Events', 'OR', 'CI low', 'CI high', 'p'])
s14['p'] = s14['p'].map(lambda v: '–' if pd.isna(v) else f'{v:.2e}' if v < 0.001 else f'{v:.3f}')
pd.set_option('display.width', 200); print(s14.to_string(index=False, formatters={'Events': '{:,}'.format, 'OR': '{:.3f}'.format,
      'CI low': '{:.3f}'.format, 'CI high': '{:.3f}'.format}))

                                                  Specification Events    OR CI low CI high        p
                                      All events, no clustering 21,014 1.279  1.185   1.379 2.09e-10
                        All events, clustered by source account 21,014 1.279  1.018   1.606    0.035
                        All events, clustered by target account 21,014 1.279  1.124   1.454 1.83e-04
              All events, two-way clustered (source and target) 21,014 1.279  1.011   1.617    0.040
                      All events, clustered by source community 21,014 1.279  0.975   1.677    0.076
All events, cluster bootstrap over source accounts (B = 10,000) 21,014 1.279  1.024   1.614        –
  Largest connected component only, clustered by source account 16,455 1.060  0.857   1.312    0.589
         Accounts active in both periods, account fixed effects  3,659 1.134  0.942   1.364    0.183
